# Colab setup: Google Drive workspace + raw data

Run top to bottom at the start of every Colab session.

| Where | What | Why |
|---|---|---|
| `MyDrive/<DRIVE_DIR>/downloads/` | RDD2022 zip (13.3 GB) | persists across sessions, downloaded once |
| `MyDrive/<DRIVE_DIR>/raw/mwpd/` | MWPD images | small, persists |
| `MyDrive/<DRIVE_DIR>/outputs/` | logs, training runs | persists |
| `/content/deep-learning-cse365/` | repo + extracted RDD2022 + processed data | local disk is much faster than Drive for many small files |

**Drive quota:** the free plan has 15 GB. The RDD2022 zip alone is 13.3 GB.
If you do not have enough space, set `ZIP_ON_DRIVE = False`: the zip is then downloaded to local disk
every session (a few minutes on Colab) and only MWPD + outputs are kept on Drive.

In [ ]:
# ---- Settings ----
DRIVE_DIR = "cse365_pothole"   # personal folder under MyDrive
ZIP_ON_DRIVE = True            # keep the 13.3 GB RDD2022 zip on Drive (needs ~14 GB free)
REPO_URL = "https://github.com/queso-gato1355/deep-learning-cse365.git"
BRANCH = "main"

In [ ]:
# 1. Mount Drive and create the personal folder
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_DIR
for sub in ["downloads/rdd2022", "raw/mwpd", "outputs/logs", "outputs/runs"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
print("Drive workspace:", DRIVE_ROOT)

In [ ]:
# 2. Clone (or update) the repo on local disk
import subprocess

REPO = Path("/content/deep-learning-cse365")
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
%cd {REPO}
!git log --oneline -3

In [ ]:
# 3. Dependencies for the data pipeline.
# Colab already ships numpy/pandas/opencv/matplotlib/seaborn/requests/pyyaml/tqdm.
# requirements.txt pins Python 3.11 + CUDA 12.8 wheels, so it is NOT installed wholesale here.
!pip install -q albumentations==2.0.8 ruff==0.17.3

In [ ]:
# 4. Link persistent Drive folders into the repo layout expected by configs/data.yaml
import os
import shutil

def link(target: Path, link_path: Path) -> None:
    """Replace ``link_path`` with a symlink to ``target``."""
    link_path.parent.mkdir(parents=True, exist_ok=True)
    if link_path.is_symlink():
        link_path.unlink()
    elif link_path.exists():
        shutil.rmtree(link_path)
    os.symlink(target, link_path)

LOCAL_DOWNLOADS = Path("/content/downloads")
ZIP_DIR = (DRIVE_ROOT / "downloads") if ZIP_ON_DRIVE else LOCAL_DOWNLOADS
(ZIP_DIR / "rdd2022").mkdir(parents=True, exist_ok=True)

link(ZIP_DIR, REPO / "data/raw/_downloads")
link(DRIVE_ROOT / "raw/mwpd", REPO / "data/raw/mwpd")
link(DRIVE_ROOT / "outputs/logs", REPO / "outputs/logs")
link(DRIVE_ROOT / "outputs/runs", REPO / "outputs/runs")
# data/raw/rdd2022 and data/processed stay on local disk (fast I/O).
!ls -la data/raw outputs

In [ ]:
# 5. Download the RDD2022 zip (resumable, md5-verified). Skipped if already complete.
import hashlib
import requests
from tqdm.auto import tqdm

RDD_URL = "https://ndownloader.figshare.com/files/38030910"
RDD_ZIP = ZIP_DIR / "rdd2022" / "RDD2022_released_through_CRDDC2022.zip"
RDD_SIZE = 13_264_172_619
RDD_MD5 = "b62bd51d2ffcfaa76c60f234f0cc2bb3"

def md5sum(path: Path) -> str:
    h = hashlib.md5()
    with path.open("rb") as f, tqdm(total=path.stat().st_size, unit="B", unit_scale=True, desc="md5") as bar:
        for chunk in iter(lambda: f.read(1 << 24), b""):
            h.update(chunk)
            bar.update(len(chunk))
    return h.hexdigest()

part = RDD_ZIP.with_suffix(".zip.part")
if RDD_ZIP.exists() and RDD_ZIP.stat().st_size == RDD_SIZE:
    print("Zip already present:", RDD_ZIP)
else:
    free = shutil.disk_usage(ZIP_DIR).free
    have = part.stat().st_size if part.exists() else 0
    if free < RDD_SIZE - have:
        raise RuntimeError(f"Only {free/1e9:.1f} GB free at {ZIP_DIR}; need {(RDD_SIZE-have)/1e9:.1f} GB. "
                           "Free up Drive space or set ZIP_ON_DRIVE = False.")
    headers = {"Range": f"bytes={have}-"} if have else {}
    with requests.get(RDD_URL, headers=headers, stream=True, timeout=60) as r:
        r.raise_for_status()
        mode = "ab" if have and r.status_code == 206 else "wb"
        start = have if mode == "ab" else 0
        with part.open(mode) as f, tqdm(total=RDD_SIZE, initial=start, unit="B", unit_scale=True, desc="RDD2022") as bar:
            for chunk in r.iter_content(chunk_size=1 << 24):
                f.write(chunk)
                bar.update(len(chunk))
    if md5sum(part) != RDD_MD5:
        part.unlink()
        raise ValueError("md5 mismatch: partial file deleted, re-run this cell.")
    part.replace(RDD_ZIP)
    print("Downloaded:", RDD_ZIP)

In [ ]:
# 6. Download MWPD (to Drive, SHA-256 verified, skips files already there)
#    and extract RDD2022 to local disk (/content/.../data/raw/rdd2022, ~13 GB, takes a while).
!python -m src.data.download --dataset all
!du -sh data/raw/rdd2022 data/raw/mwpd/ 2>/dev/null

## Optional: run the Phase-1 pipeline

Rebuilds `data/processed/` (local disk) and the EDA figures/tables.
`split.py` regenerates the split lists deterministically (seed 42), overwriting the committed ones;
the `git status` afterwards should show no changes under `data/processed/splits/`.

In [ ]:
!make data PY=python
!git status --short data/processed/splits outputs/tables

## Disk check

In [ ]:
!df -h /content /content/drive | cat